In [1]:
import torch
from torch import nn, tensor
import torch.nn.functional as F

### Simple

In [2]:
x = torch.tensor([[3.0]])
w1 = torch.tensor(2.0, requires_grad=True)

In [3]:
y = x ** w1

In [5]:
loss = y**3 - 5

In [6]:
loss.backward()

In [7]:
y

tensor([[9.]], grad_fn=<PowBackward1>)

In [8]:
loss.item()

724.0

In [9]:
w1.grad

tensor(2402.6650)

In [10]:
dL_dy = 3 * (y.item()) ** 2

In [11]:
dy_dw1 = (x ** w1 * torch.log(x)).item()

In [12]:
dL_dw1 = dL_dy * dy_dw1

In [13]:
dL_dw1

2402.665002822876

### Non Detach

In [23]:
x = torch.tensor([[3.0]])
w1 = torch.tensor(2.0, requires_grad=True)

In [24]:
y = x * w1 + (x * w1 - 10)
y

tensor([[2.]], grad_fn=<AddBackward0>)

In [25]:
loss = y**3 - 5
loss.backward()

In [26]:
w1.grad

tensor(72.)

In [27]:
dL_dy = 3 * y ** 2
dy_dw1 = 2 * x
dL_dw1 = dL_dy * dy_dw1

In [28]:
dL_dw1

tensor([[72.]], grad_fn=<MulBackward0>)

### Detach

In [17]:
x = torch.tensor([[3.0]])
w1 = torch.tensor(2.0, requires_grad=True)

In [18]:
y = x * w1 + (x * w1 - 10).detach()
y

tensor([[2.]], grad_fn=<AddBackward0>)

In [19]:
loss = y**3 - 5
loss.backward()

In [20]:
w1.grad

tensor(36.)

In [21]:
dL_dy = 3 * y ** 2
dy_dw1 = x
dL_dw1 = dL_dy * dy_dw1

In [22]:
dL_dw1

tensor([[36.]], grad_fn=<MulBackward0>)

### 2 Related Param

In [44]:
a = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([3.0], requires_grad=True)

In [45]:
x1 = a * 2 + b ** 2
x2 = b * 3 + a ** (0.5)

In [46]:
x1, x2

(tensor([13.], grad_fn=<AddBackward0>),
 tensor([10.4142], grad_fn=<AddBackward0>))

In [47]:
loss_type_1 = F.mse_loss(x1.detach(), x2)

In [48]:
loss_type_1.backward()

In [49]:
dx1_da = 2
dx1_db = 2 * b

dx2_da = 0.5 * a ** (-0.5)
dx2_db = 3 

In [50]:
dL1_dx1 = 0
dL1_dx2 = 2 * (x2 - x1)

dL1_da = dL1_dx1 * dx1_da + dL1_dx2 * dx2_da
dL1_db = dL1_dx1 * dx1_db + dL1_dx2 * dx2_db

In [51]:
dL1_da, dL1_db

(tensor([-1.8284], grad_fn=<AddBackward0>),
 tensor([-15.5147], grad_fn=<AddBackward0>))

In [52]:
a.grad, b.grad

(tensor([-1.8284]), tensor([-15.5147]))

In [53]:
loss_type_2 = F.mse_loss(x1, x2.detach())

In [54]:
loss_type_2.backward()

In [55]:
dL2_dx1 = 2 * (x1 - x2)
dL2_dx2 = 0

dL2_da = dL2_dx1 * dx1_da + dL2_dx2 * dx2_da
dL2_db = dL2_dx1 * dx1_db + dL2_dx2 * dx2_db

In [56]:
a.grad, b.grad

(tensor([8.5147]), tensor([15.5147]))

In [57]:
dL1_da + dL2_da, dL1_db + dL2_db

(tensor([8.5147], grad_fn=<AddBackward0>),
 tensor([15.5147], grad_fn=<AddBackward0>))